# Tesla Valve

Example provided by Philip Lederer.

The original drawing by N. Tesla in his patent (US patent 1329559 "Valvular Conduit"):

<img src="figures/tesla_patent.png" alt="tesla" width="650" align="center"/>

The valve has no moving parts: flow from left to right is deflected into the loops and blocks itself, flow from right to left passes almost freely.

In [29]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import ipywidgets as widgets

The parametric geometry is built with the OCC workplane (expand the cell to see the code):

In [30]:
def GetValve(N, dim, R = 1.4, alpha = 45, Winlet = 1, beta = 190, L1 = 11, L2 = 6, Linlet = 5, closevalve = False):
    alphar = alpha * pi/180
    W = 1/cos(alphar/2)
    facW = 1
    Winlet2 =  Winlet
        
    wp = WorkPlane()
    p1 = 0
    p2 = 0

    wp.MoveTo(p1,p1) 
    r1 = Rectangle(L1,W).Face()
    
    r2 = wp.MoveTo(p1,p2+W).Rotate(-90+alpha).Move(W).Rotate(90).Rectangle(L2,W).Face()
    
    wp.MoveTo(p1,p2+W).Move(L2)
    c1 = Face(wp.Arc(W + R, -beta).Line(L1).Rotate(-90).Line(W).Rotate(-90).Line(L1).Arc(R,beta).Close().Wire())
    wp.MoveTo(0,W).Direction(1,0)
    cutplane = Rectangle(2*L1,4*L1).Face()
    

    v1 = r1 + r2  + (cutplane*c1) 
    
    ll = L1 + L1 * cos(alphar) - cos(alphar) * W
    hh = L1 * sin(alphar) - (1-sin(alphar)) * W
    didi = sqrt((L1 + L1 * cos(alphar))**2 + (L1 * sin(alphar))**2 ) - 2*W * sin(alphar/2) 
    
    dd = gp_Dir(cos(alpha), sin(alpha),0)
    v2 = v1.Mirror(Axis((0,0,0), X)).Move((0,W,0)).Rotate(Axis((0,0,0), Z), alpha).Move((L1,0,0))

    onevalve = (v1 + v2.Reversed()).Move((0,-W/2,0)).Rotate(Axis((0,0,0), Z), -alpha/2)

    vv = onevalve
    
    for i in range(1,N):
        vv = onevalve.Move((didi * i ,0, 0)) + vv
        
    
    
    inlet = wp.MoveTo(-Linlet,-Winlet2/2).Rectangle(Linlet* facW + W * sin(alphar/2) ,Winlet2).Face()
    outlet = wp.MoveTo(didi*N + W/2 * sin(alphar/2)* facW,-Winlet2/2).Rectangle(Linlet ,Winlet2).Face()
    vv = inlet + vv + outlet
    
    if closevalve == True:
        c1_x = -Linlet
        c1_y = -Winlet2/2

        c2_x = didi*N + W/2 * sin(alphar/2)* facW + Linlet
        c2_y = -Winlet2/2
        
        wp.MoveTo(c1_x, c1_y).Direction(-1,0)
        R2 = 3
        LL = c2_x - c1_x
        # close = Face(wp.Arc(R2+Winlet2,180).Line(LL).Arc(R2+Winlet2,180).Rotate(90).Line(Winlet2).Rotate(90).Arc(R2,-180).Line(LL).Arc(R2,-180).Close().Wire())
        
        close = Face(wp.Arc(R2+Winlet2,-180).Line(LL).Arc(R2+Winlet2,-180).Rotate(-90).Line(Winlet2).Rotate(-90).Arc(R2,180).Line(LL).Arc(R2,180).Close().Wire()).Reversed()
        # close.face.name

    teslavalve = vv
    # vv = onevalve
    if dim == 3:
        teslavalve = vv.Extrude( Winlet*Z )
        teslavalve.faces.name="wall"
        
    
        if closevalve == False:
            teslavalve.faces.Min(X).name="inlet"
            teslavalve.faces.Max(X).name="outlet"
            teslavalve.faces.Min(X).Identify(teslavalve.faces.Max(X),"periodic", IdentificationType.PERIODIC)

    else:
        teslavalve.edges.name="wall"
        teslavalve.faces.name="valve"
        teslavalve.edges.Min(X).name="inlet"
        teslavalve.edges.Max(X).name="outlet"
        
        if closevalve == True:
            close.edges.name = "wall"
            teslavalve =  Glue([teslavalve, close])
    
        if closevalve == False:        
            teslavalve.edges.Min(X).Identify(teslavalve.edges.Max(X),"periodic", IdentificationType.PERIODIC)

    return teslavalve

In [31]:
valve = GetValve(N = 3, dim = 2, R = 0.5, alpha = 25, Winlet = 1, beta = 180, L1 = 6.4, L2 = 7, Linlet = 5, closevalve = True)
mesh = valve.GenerateMesh(dim=2, maxh=0.5).Curve(3)
Draw(mesh);

## Incompressible Navier-Stokes equations

$$
\frac{\partial u}{\partial t} + (u \cdot \nabla) u - \nu \Delta u + \nabla p = f, \qquad \operatorname{div} u = 0
$$

A volume force $f$ drives the flow either to the left or to the right; we compare the mass flow.

In [ ]:
nu = 0.005   # viscosity
dt = 0.002   # time step
tend = 50    # end time
order = 2    # fem order (velocity)

We use the **mass conserving mixed stress (MCS) method** (Gopalakrishnan, Lederer, Schöberl, 2020; NGSolve Technical Note 2026-03).
Besides velocity and pressure, the viscous stress $\sigma = 2 \nu \operatorname{dev} \varepsilon(u)$ is an unknown:

* velocity $u$: $BDM^k$, normal-continuous. The pressure space is exactly $\operatorname{div} V_h$, so $u$ is exactly divergence-free
* stress $\sigma$: trace-free matrices with continuous normal-tangential component $\sigma_{nt}$. It enforces the tangential continuity of $u$ weakly
* tangential velocity $\hat u$ on facets: hybridization, which makes the stress element-local
* skew-symmetric multiplier $\eta$: weak symmetry of $\sigma$

All spaces are created from the velocity space. Stress and multiplier are element-local: as private spaces they are eliminated during assembly, and only $u$, $\hat u$ and $p$ remain as global unknowns:

In [ ]:
V    = HDiv(mesh, order=order, dirichlet="wall")      # velocity
S    = Discontinuous(V.CreateStressSpace())            # stress, hybridized
Vhat = S.CreateMultiplierSpace(dirichlet="wall")       # tangential velocity
Q    = V.CreateDivergenceSpace()                       # pressure = div V
W    = S.base_space.CreateSymmetryMultiplierSpace()    # skew-symmetric multiplier

# private spaces are eliminated element by element
X = V * PrivateSpace(S) * Vhat * Q * PrivateSpace(W)
print ("ndof:", X.ndof)
(u, sigma, uhat, p, eta), (v, tau, vhat, q, xi) = X.TnT()

n = specialcf.normal(mesh.dim)

def tang(u): # tangential part of a vector field
    return u - (u*n)*n

Implicit/explicit time-stepping, with the Stokes operator implicit and the convection explicit:

$$
\frac{u_{n+1}-u_n}{\Delta t} - \operatorname{div} \sigma_{n+1} + \nabla p_{n+1} = f - (u_n \cdot \nabla) u_n, \qquad
\operatorname{div} u_{n+1} = 0
$$

The Stokes operator in mixed form: find $(u, \sigma, \hat u, p, \eta)$ such that

$$
\begin{aligned}
-\tfrac{1}{2\nu} (\sigma, \tau) + b(\tau, (u, \hat u)) + (\tau, \eta) & = 0 \\
b(\sigma, (v, \hat v)) - (\operatorname{div} v, p) & = (f, v) \\
(\operatorname{div} u, q) = 0, \qquad (\sigma, \xi) & = 0
\end{aligned}
$$

for all test functions, with the pairing of stresses and velocities

$$
b(\tau, (v, \hat v)) = \sum_T \Big( \int_T \tau : \nabla v \, dx - \int_{\partial T} \tau_{nt} \cdot (v - \hat v)_t \, ds \Big)
$$

Stokes operator, modified mass matrix, and right hand side:

In [ ]:
dS = dx(element_boundary = True)

def B(tau, v, vhat):
    return InnerProduct(tau, Grad(v))*dx - tang(tau*n)*tang(v-vhat)*dS

stokes = -1/(2*nu)*InnerProduct(sigma, tau)*dx + B(tau, u, uhat) + B(sigma, v, vhat) \
         - div(u)*q*dx - div(v)*p*dx - 1e-10/nu*p*q*dx \
         + InnerProduct(sigma, xi)*dx + InnerProduct(tau, eta)*dx

K = BilinearForm(stokes, symmetric=True).Assemble()
mstar = BilinearForm(u*v*dx + dt*stokes, symmetric=True).Assemble()
inv = mstar.mat.Inverse(X.FreeDofs(), inverse = "sparsecholesky")

f = LinearForm(CF((1,0)) * v * dx(definedon = mesh.Materials("valve"))).Assemble()

Convective term with upwinding on the facets, applied matrix-free.
The upwind value is the tangential velocity on the facets with the full order $k$, one more than the MCS variable $\hat u$. It gets its own facet space:

In [ ]:
Vup = TangentialFacetFESpace(mesh, order=order, dirichlet="wall")
Y = V * Vup
(uc, uup), (vc, vup) = Y.TnT()

conv = BilinearForm(Y, nonlinear_matrix_free_bdb=True)
conv += -InnerProduct(grad(vc)*uc, uc) * dx(bonus_intorder=order)

un = uc.Operator("normalcomponent")
uup_t = uup.Operator("tangentialcomponent")
vn = vc.Operator("normalcomponent")
conv += IfPos(un, un * uc * vc, 2 * un * un * vn + un * ((2*uup_t-uc) * vc) ) * dx(element_boundary = True, bonus_intorder=order)
conv.Assemble()

Averaging of tangential components to the facet space, and the convection as an operator on the product space:

In [ ]:
u1, v1 = V.TnT()
uh1, vh1 = Vup.TnT()

massfacet = BilinearForm(tang(uh1)*tang(vh1)*dx(element_boundary=True)).Assemble()
mixedmassfacet = BilinearForm(u1*tang(vh1)*dx(element_boundary=True)).Assemble()
invfacet = massfacet.mat.CreateBlockSmoother(Vup.CreateSmoothingBlocks(blocktype="facet"))

avg_op = Y.embeddings[0] + Y.embeddings[1]@invfacet@mixedmassfacet.mat
eu, ru = X.embeddings[0], X.restrictions[0]
conv_operator = eu @ Y.restrictions[0] @ conv.mat @ avg_op @ ru

In [ ]:
gfu = GridFunction(X)

def TimeStep(factor=1, callback = lambda: None):
    t = 0
    cnt = 0
    with TaskManager():
        while t < tend:
          res = factor*f.vec - conv_operator * gfu.vec - K.mat * gfu.vec
          gfu.vec.data += dt * inv * res    
          
          t += dt
          cnt += 1
          callback (cnt, gfu)

In [ ]:
massflux = []
massflux_reverse = []
time = []
gfut = GridFunction(gfu.components[0].space, multidim=0)
gfut_reverse = GridFunction(gfu.components[0].space, multidim=0)


scene = Draw(gfu.components[0], mesh, min=0, max=4, autoscale=False, vectors = {"grid_size": 100})
tw = widgets.Text(value='t = 0')
display(tw)

def cbforward (cnt, gfu):
    if cnt % 100 == 0: 
        scene.Redraw()
        tw.value = "t = "+str(cnt*dt)
    if cnt % 500 == 0:
        gfut.AddMultiDimComponent(gfu.components[0].vec)
        massflux.append(abs(Integrate(gfu.components[0]*n, mesh.Boundaries("inlet"))))
        time.append(cnt*dt)

def cbreverse (cnt, gfu):
    if cnt % 100 == 0: 
        scene.Redraw()
        tw.value = "t = "+str(cnt*dt)
    if cnt % 500 == 0:
        gfut_reverse.AddMultiDimComponent(gfu.components[0].vec)
        massflux_reverse.append(abs(Integrate(gfu.components[0]*n, mesh.Boundaries("inlet"))))



gfu.vec[:] = 0
TimeStep (1, cbforward)

gfu.vec[:] = 0
TimeStep (-1, cbreverse)

Flow from left to right activates the valve: the fluid blocks itself and the flow becomes unsteady.
From right to left the flow is nearly laminar and much faster. Solutions over time:

In [ ]:
Draw (gfut, mesh, interpolate_multidim=True, animate=True,
      min=0, max=4, autoscale=False, vectors = {"grid_size": 100});
Draw (gfut_reverse, mesh, interpolate_multidim=True, animate=True,
      min=0, max=6, autoscale=False, vectors = {"grid_size": 100});

The mass flux over time:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(time, massflux, "-", color = "orange", label = "left to right (blocking)")
plt.plot(time, massflux_reverse, "-", color = "blue", label = "right to left (passing)")
plt.xlabel("time"); plt.ylabel("mass flux")
plt.legend(loc = "lower right")
plt.show()

## The same method at industrial scale

Flow through the fan of a heat pump, with rotating and stationary parts:

<video src="airfoil.mp4" width="100%" controls loop muted></video>

* 18 million unknowns, $H(\operatorname{div})$-conforming velocity with the mass conserving mixed stress (MCS) formulation
* operator splitting in time (IPCS), the fan rotates by 1 degree per time step
* 1 minute per time step on one cluster node with an A100 GPU